# 🧠 Module 4.5: Q-Learning for Image Enhancement

## Off-Policy Control: Learning the Optimal Policy Directly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_04_Basic_RL_Algorithms/05_Q_Learning/q_learning.ipynb)

---

## 🎯 Learning Objectives
1. Derive the **Q-learning** update rule from the Bellman optimality equation
2. Understand **off-policy** learning: behavior policy ≠ target policy
3. Sketch the **convergence proof** via stochastic approximation
4. Compare Q-learning vs SARSA (off-policy vs on-policy) on the Cliff Walking problem
5. Address **maximization bias** with Double Q-learning
6. Apply tabular Q-learning to image enhancement with a small discrete action space

**Prerequisites:** SARSA and TD Learning (Module 4.4)

---

In [ ]:
# Setup - Install dependencies (Google Colab)
!pip install numpy matplotlib opencv-python-headless pillow scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset
import torchvision
import numpy as np

# CIFAR-10 for image filter selection environment
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
# Take small subset for fast RL experiments
real_images = [np.array(cifar10[i][0]) for i in range(200)]
print(f"✅ CIFAR-10 loaded: Using {len(real_images)} real photos for RL environment")
print(f"   Image shape: {real_images[0].shape}")
print("   These will be our 'states' that the RL agent processes!")

## Deep Derivation: Q-Learning Convergence Proof

### Step 1: Q-Learning Update Rule
$$Q_{t+1}(s_t, a_t) = Q_t(s_t, a_t) + \alpha_t(s_t, a_t)\left[r_t + \gamma \max_{a'} Q_t(s_{t+1}, a') - Q_t(s_t, a_t)\right]$$

### Step 2: Stochastic Approximation Framework
Q-learning is a special case of the **Robbins-Monro** algorithm:
$$x_{n+1} = x_n + \alpha_n [f(x_n) + M_n]$$

where $f(x_n) = \mathcal{T}Q(s,a) - Q(s,a)$ and $M_n$ is zero-mean noise.

### Step 3: Convergence Conditions (Robbins-Monro)
Q-learning converges to $Q^*$ with probability 1 if:
1. $\sum_{t=0}^\infty \alpha_t(s,a) = \infty$ (visit every state-action infinitely often)
2. $\sum_{t=0}^\infty \alpha_t^2(s,a) < \infty$ (step sizes decrease fast enough)
3. $\text{Var}[r + \gamma \max_{a'} Q(s',a') | s, a] < \infty$ (bounded variance)

### Step 4: Why It Works (Intuition via ODE Method)
In the limit of small $\alpha$, the Q-learning trajectory approximates the ODE:
$$\dot{Q}(s,a) = \mathcal{T}Q(s,a) - Q(s,a) = E[r + \gamma \max_{a'} Q(s',a') | s, a] - Q(s,a)$$

The unique fixed point is $Q^*$ because $\mathcal{T}$ is a contraction (proved above).

### Step 5: Sample Complexity
Expected number of samples to achieve $\|Q_T - Q^*\|_\infty \leq \epsilon$:
$$T = O\left(\frac{|S||A|}{\epsilon^2(1-\gamma)^5} \log\frac{|S||A|}{\delta}\right)$$

### Step 6: Maximization Bias (Why Double Q-Learning Helps)

**Problem:** $E[\max_a Q(s,a)] \geq \max_a E[Q(s,a)]$ (Jensen's inequality)

**Proof of bias:**
Let $Q(s,a) = Q^*(s,a) + \epsilon_a$ where $\epsilon_a$ are zero-mean noise terms.
$$E[\max_a Q(s,a)] = E[\max_a (Q^*(s,a) + \epsilon_a)] \geq \max_a Q^*(s,a)$$

Equality holds only if all $\epsilon_a = 0$. Otherwise: **systematic overestimation!**

**Double Q-learning fix:**
Use one Q to select: $a^* = \arg\max_a Q_1(s,a)$
Use other Q to evaluate: target $= r + \gamma Q_2(s, a^*)$

Since $Q_1$ and $Q_2$ have independent noise, the bias cancels.

## 1. From SARSA to Q-Learning: The Off-Policy Insight

### 1.1 SARSA Recap (On-Policy)

SARSA updates $Q$ toward the value of the **action actually taken** under the current policy:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)\right]$$

Here $A_{t+1}$ is chosen by the **same** $\epsilon$-greedy policy. SARSA learns $q_\pi$ for the current policy $\pi$.

### 1.2 The Key Idea: Use the Max

Q-learning replaces $Q(S_{t+1}, A_{t+1})$ with $\max_a Q(S_{t+1}, a)$:

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)\right]$$

This means Q-learning directly approximates $q_*$ (the **optimal** action-value function), regardless of the policy being followed!

### 1.3 Why This is Off-Policy

- **Behavior policy** (what the agent actually does): $\epsilon$-greedy or any exploratory policy
- **Target policy** (what is being learned): greedy policy $\pi_*(s) = \arg\max_a Q(s,a)$

These are **different** — hence off-policy. The behavior policy just needs to explore enough.

## 2. Q-Learning — Complete Derivation

### 2.1 Starting Point: Bellman Optimality Equation

The optimal action-value function $q_*$ satisfies:

$$q_*(s, a) = \mathbb{E}\left[R_{t+1} + \gamma \max_{a'} q_*(S_{t+1}, a') \mid S_t = s, A_t = a\right]$$

### 2.2 Stochastic Approximation

We can't compute the expectation directly (unknown model), so we use **sampled transitions**.

Given a sample $(S_t, A_t, R_{t+1}, S_{t+1})$, the **sample Bellman optimality backup** is:

$$R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a')$$

### 2.3 The Update Rule

Apply the standard stochastic approximation formula $x_{n+1} = x_n + \alpha_n(\text{target} - x_n)$:

$$\boxed{Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t)\right]}$$

### 2.4 TD Error for Q-Learning

$$\delta_t = R_{t+1} + \gamma \max_{a'} Q(S_{t+1}, a') - Q(S_t, A_t)$$

Compare with SARSA: $\delta_t^{\text{SARSA}} = R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)$

The only difference is $\max_{a'} Q(S_{t+1}, a')$ vs $Q(S_{t+1}, A_{t+1})$.

### 2.5 Full Algorithm

$$\boxed{\textbf{Q-Learning (Off-Policy TD Control)}}$$

Initialize $Q(s, a)$ arbitrarily (terminal states: $Q(s_{\text{term}}, \cdot) = 0$)

**Repeat** for each episode:
1. Initialize $S$
2. **For** each step:
   - Choose $A$ from $S$ using policy derived from $Q$ (e.g., $\epsilon$-greedy)
   - Take action $A$, observe $R, S'$
   - $Q(S, A) \leftarrow Q(S, A) + \alpha[R + \gamma \max_a Q(S', a) - Q(S, A)]$
   - $S \leftarrow S'$
3. **Until** $S$ is terminal

## 3. Convergence Proof Sketch

### 3.1 Stochastic Approximation Framework

Q-learning is an instance of **stochastic approximation** of the form:

$$Q_{t+1}(s, a) = (1 - \alpha_t(s,a)) Q_t(s, a) + \alpha_t(s,a) \left[R_{t+1} + \gamma \max_{a'} Q_t(S_{t+1}, a')\right]$$

### 3.2 Conditions for Convergence

**Theorem (Watkins & Dayan, 1992):** Q-learning converges to $q_*$ with probability 1 if:

1. **State-action coverage:** All $(s, a)$ pairs are visited infinitely often
2. **Robbins-Monro step sizes:**
$$\sum_{t=1}^{\infty} \alpha_t(s, a) = \infty \quad \text{and} \quad \sum_{t=1}^{\infty} \alpha_t^2(s, a) < \infty$$
3. Rewards have **bounded variance**

### 3.3 Proof Sketch via Contraction

Define the **Bellman optimality operator** $T_*$:

$$(T_* Q)(s, a) = \mathbb{E}[R + \gamma \max_{a'} Q(S', a') | s, a]$$

$T_*$ is a **$\gamma$-contraction** in the max-norm:

$$\|T_* Q_1 - T_* Q_2\|_\infty \leq \gamma \|Q_1 - Q_2\|_\infty$$

**Proof of contraction:**

$$|(T_* Q_1)(s,a) - (T_* Q_2)(s,a)| = \gamma \left|\mathbb{E}\left[\max_{a'} Q_1(S',a') - \max_{a'} Q_2(S',a')\right]\right|$$

Since $|\max_a f(a) - \max_a g(a)| \leq \max_a |f(a) - g(a)|$:

$$\leq \gamma \mathbb{E}\left[\max_{a'} |Q_1(S',a') - Q_2(S',a')|\right] \leq \gamma \|Q_1 - Q_2\|_\infty$$

By the **Banach fixed-point theorem**, $T_*$ has a unique fixed point $q_*$.

The stochastic approximation theory then guarantees that the noisy, sampled updates converge to this fixed point under the Robbins-Monro conditions. $\blacksquare$

### 3.4 Practical Implications

- A constant step size $\alpha$ violates condition 2 but works well for non-stationary problems
- $\epsilon$-greedy with $\epsilon_k = 1/k$ satisfies condition 1
- Convergence can be slow — $O(1/\alpha)$ steps to reduce error by constant factor

In [ ]:
class QLearningAgent:
    """Tabular Q-Learning agent."""

    def __init__(self, n_actions, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q = defaultdict(lambda: np.zeros(n_actions))

    def get_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])

    def update(self, state, action, reward, next_state, done):
        td_target = reward + self.gamma * np.max(self.Q[next_state]) * (1 - done)
        td_error = td_target - self.Q[state][action]
        self.Q[state][action] += self.alpha * td_error
        return td_error

    def get_policy(self):
        policy = {}
        for state in self.Q:
            policy[state] = np.argmax(self.Q[state])
        return policy

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

class SARSAAgent:
    """Tabular SARSA agent for comparison."""

    def __init__(self, n_actions, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q = defaultdict(lambda: np.zeros(n_actions))

    def get_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        return np.argmax(self.Q[state])

    def update(self, state, action, reward, next_state, next_action, done):
        td_target = reward + self.gamma * self.Q[next_state][next_action] * (1 - done)
        td_error = td_target - self.Q[state][action]
        self.Q[state][action] += self.alpha * td_error
        return td_error

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

print("Agent classes defined.")

## 4. Q-Learning vs SARSA: The Cliff Walking Example

The classic **Cliff Walking** environment illustrates the fundamental difference between on-policy (SARSA) and off-policy (Q-learning) methods.

- SARSA (on-policy) learns a **safe** path that avoids the cliff because it accounts for its own exploratory behavior
- Q-learning (off-policy) learns the **optimal** path along the cliff edge, since it optimizes assuming greedy execution

In [ ]:
class CliffWalkingEnv:
    """Classic Cliff Walking environment.

    4x12 grid. Start at bottom-left, goal at bottom-right.
    Bottom row (except start/goal) is the cliff: -100 reward and reset.
    All other moves: -1 reward.
    """

    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)
        self.goal = (3, 11)
        self.cliff = [(3, c) for c in range(1, 11)]
        self.n_actions = 4  # up, right, down, left
        self.action_deltas = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        r, c = self.state
        dr, dc = self.action_deltas[action]
        nr, nc = max(0, min(self.rows - 1, r + dr)), max(0, min(self.cols - 1, c + dc))
        self.state = (nr, nc)

        if self.state in self.cliff:
            self.state = self.start
            return self.start, -100, False
        elif self.state == self.goal:
            return self.state, -1, True
        else:
            return self.state, -1, False

cliff_env = CliffWalkingEnv()

# Train both agents
n_episodes = 500
n_runs = 50

sarsa_rewards_all = np.zeros((n_runs, n_episodes))
qlearn_rewards_all = np.zeros((n_runs, n_episodes))

for run in range(n_runs):
    sarsa = SARSAAgent(4, alpha=0.5, gamma=1.0, epsilon=0.1,
                       epsilon_min=0.1, epsilon_decay=1.0)
    qlearn = QLearningAgent(4, alpha=0.5, gamma=1.0, epsilon=0.1,
                            epsilon_min=0.1, epsilon_decay=1.0)

    for ep in range(n_episodes):
        # SARSA episode
        state = cliff_env.reset()
        action = sarsa.get_action(state)
        done = False
        ep_reward = 0
        while not done:
            next_state, reward, done = cliff_env.step(action)
            next_action = sarsa.get_action(next_state)
            sarsa.update(state, action, reward, next_state, next_action, done)
            state, action = next_state, next_action
            ep_reward += reward
        sarsa_rewards_all[run, ep] = ep_reward

        # Q-learning episode
        state = cliff_env.reset()
        done = False
        ep_reward = 0
        while not done:
            action = qlearn.get_action(state)
            next_state, reward, done = cliff_env.step(action)
            qlearn.update(state, action, reward, next_state, done)
            state = next_state
            ep_reward += reward
        qlearn_rewards_all[run, ep] = ep_reward

# Final policies for visualization (from last run)
sarsa_policy = {s: np.argmax(sarsa.Q[s]) for s in sarsa.Q}
qlearn_policy = {s: np.argmax(qlearn.Q[s]) for s in qlearn.Q}

print("✅ Training complete!")

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Learning curves
window = 20
sarsa_mean = sarsa_rewards_all.mean(axis=0)
qlearn_mean = qlearn_rewards_all.mean(axis=0)
sarsa_smooth = np.convolve(sarsa_mean, np.ones(window)/window, mode='valid')
qlearn_smooth = np.convolve(qlearn_mean, np.ones(window)/window, mode='valid')

axes[0].plot(sarsa_smooth, 'b-', linewidth=2, label='SARSA (on-policy)')
axes[0].plot(qlearn_smooth, 'r-', linewidth=2, label='Q-Learning (off-policy)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Reward per Episode')
axes[0].set_title('Cliff Walking: SARSA vs Q-Learning\n(Higher is better)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-200, 0)

# Visualize SARSA path
def plot_cliff_policy(policy, env, title, ax):
    grid = np.zeros((env.rows, env.cols))
    arrow_dx = [0, 0.3, 0, -0.3]
    arrow_dy = [-0.3, 0, 0.3, 0]

    # Color the cliff
    for (r, c) in env.cliff:
        grid[r, c] = -1
    grid[env.start] = 0.5
    grid[env.goal] = 1.0

    ax.imshow(grid, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')

    for r in range(env.rows):
        for c in range(env.cols):
            state = (r, c)
            if state == env.goal:
                ax.text(c, r, 'G', ha='center', va='center', fontsize=12, fontweight='bold')
            elif state == env.start:
                ax.text(c, r, 'S', ha='center', va='center', fontsize=12, fontweight='bold')
            elif state in env.cliff:
                ax.text(c, r, '☠', ha='center', va='center', fontsize=10)
            elif state in policy:
                a = policy[state]
                ax.arrow(c, r, arrow_dx[a], arrow_dy[a],
                        head_width=0.15, head_length=0.1, fc='black', ec='black')

    ax.set_title(title)
    ax.set_xticks(range(env.cols))
    ax.set_yticks(range(env.rows))

plot_cliff_policy(sarsa_policy, cliff_env, 'SARSA Policy (Safe Path)', axes[1])
plot_cliff_policy(qlearn_policy, cliff_env, 'Q-Learning Policy (Optimal Path)', axes[2])

plt.tight_layout()
plt.show()

print("\nSARSA learns the SAFE path (away from cliff) due to ε-exploration.")
print("Q-learning learns the OPTIMAL path (along cliff edge) assuming greedy execution.")
print(f"\nSARSA avg reward (last 100 ep):   {sarsa_mean[-100:].mean():.1f}")
print(f"Q-learning avg reward (last 100 ep): {qlearn_mean[-100:].mean():.1f}")

## 5. Maximization Bias and Double Q-Learning

### 5.1 The Problem: Maximization Bias

Q-learning uses $\max_a Q(S', a)$ as the target. This introduces a **positive bias**:

$$\mathbb{E}\left[\max_a Q(s, a)\right] \geq \max_a \mathbb{E}[Q(s, a)]$$

This is a consequence of **Jensen's inequality** (max is convex).

### 5.2 Example of Maximization Bias

Consider two independent estimators $Q_1, Q_2$ with $\mathbb{E}[Q_1] = \mathbb{E}[Q_2] = 0$.

$$\mathbb{E}[\max(Q_1, Q_2)] \geq \max(\mathbb{E}[Q_1], \mathbb{E}[Q_2]) = 0$$

The max overestimates the true value!

### 5.3 Double Q-Learning Solution

Maintain **two** independent Q-tables: $Q_1$ and $Q_2$.

Use one to **select** the best action, the other to **evaluate** it:

With probability 0.5:

$$Q_1(S, A) \leftarrow Q_1(S, A) + \alpha\left[R + \gamma Q_2\left(S', \arg\max_a Q_1(S', a)\right) - Q_1(S, A)\right]$$

Otherwise:

$$Q_2(S, A) \leftarrow Q_2(S, A) + \alpha\left[R + \gamma Q_1\left(S', \arg\max_a Q_2(S', a)\right) - Q_2(S, A)\right]$$

Since the selecting and evaluating Q-functions are independent, the bias cancels:

$$\mathbb{E}\left[Q_2\left(S', \arg\max_a Q_1(S', a)\right)\right] \approx q_*(S', a^*)$$

In [ ]:
class DoubleQLearningAgent:
    """Double Q-Learning agent with two Q-tables."""

    def __init__(self, n_actions, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.9995):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.Q1 = defaultdict(lambda: np.zeros(n_actions))
        self.Q2 = defaultdict(lambda: np.zeros(n_actions))

    def get_action(self, state):
        if np.random.rand() < self.epsilon:
            return np.random.randint(self.n_actions)
        combined = self.Q1[state] + self.Q2[state]
        return np.argmax(combined)

    def update(self, state, action, reward, next_state, done):
        if np.random.rand() < 0.5:
            # Update Q1 using Q2 for evaluation
            best_a = np.argmax(self.Q1[next_state])
            td_target = reward + self.gamma * self.Q2[next_state][best_a] * (1 - done)
            td_error = td_target - self.Q1[state][action]
            self.Q1[state][action] += self.alpha * td_error
        else:
            # Update Q2 using Q1 for evaluation
            best_a = np.argmax(self.Q2[next_state])
            td_target = reward + self.gamma * self.Q1[next_state][best_a] * (1 - done)
            td_error = td_target - self.Q2[state][action]
            self.Q2[state][action] += self.alpha * td_error

        return td_error

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

# Demonstrate maximization bias with a simple MDP
# State A: left leads to state B (0 reward), right leads to terminal (0 reward)
# State B: many actions with mean 0 reward but high variance (-1 to +1)
# Without bias, left and right should be equally valued
# With maximization bias, Q-learning overestimates left (via max over noisy Q(B,a))

n_actions_B = 10
n_experiments = 300
n_episodes_bias = 300

left_pct_ql = np.zeros(n_episodes_bias)
left_pct_dql = np.zeros(n_episodes_bias)

for exp in range(n_experiments):
    # Q-Learning
    Q_ql = {'A': np.zeros(2), 'B': np.zeros(n_actions_B)}
    left_count_ql = np.zeros(n_episodes_bias)

    # Double Q-Learning
    Q1_dql = {'A': np.zeros(2), 'B': np.zeros(n_actions_B)}
    Q2_dql = {'A': np.zeros(2), 'B': np.zeros(n_actions_B)}
    left_count_dql = np.zeros(n_episodes_bias)

    alpha_b = 0.1
    eps_b = 0.1

    for ep in range(n_episodes_bias):
        # Q-Learning episode
        if np.random.rand() < eps_b:
            a = np.random.randint(2)
        else:
            a = np.argmax(Q_ql['A'])

        left_count_ql[ep] = 1 if a == 0 else 0

        if a == 0:  # Left → B
            b_action = np.random.randint(n_actions_B) if np.random.rand() < eps_b else np.argmax(Q_ql['B'])
            r_b = np.random.normal(-0.1, 1.0)
            Q_ql['B'][b_action] += alpha_b * (r_b - Q_ql['B'][b_action])
            Q_ql['A'][0] += alpha_b * (0 + np.max(Q_ql['B']) - Q_ql['A'][0])
        else:  # Right → terminal
            Q_ql['A'][1] += alpha_b * (0 - Q_ql['A'][1])

        # Double Q-Learning episode
        combined = Q1_dql['A'] + Q2_dql['A']
        if np.random.rand() < eps_b:
            a = np.random.randint(2)
        else:
            a = np.argmax(combined)

        left_count_dql[ep] = 1 if a == 0 else 0

        if a == 0:  # Left → B
            combined_B = Q1_dql['B'] + Q2_dql['B']
            b_action = np.random.randint(n_actions_B) if np.random.rand() < eps_b else np.argmax(combined_B)
            r_b = np.random.normal(-0.1, 1.0)

            if np.random.rand() < 0.5:
                Q1_dql['B'][b_action] += alpha_b * (r_b - Q1_dql['B'][b_action])
                best_a_B = np.argmax(Q1_dql['B'])
                Q1_dql['A'][0] += alpha_b * (0 + Q2_dql['B'][best_a_B] - Q1_dql['A'][0])
            else:
                Q2_dql['B'][b_action] += alpha_b * (r_b - Q2_dql['B'][b_action])
                best_a_B = np.argmax(Q2_dql['B'])
                Q2_dql['A'][0] += alpha_b * (0 + Q1_dql['B'][best_a_B] - Q2_dql['A'][0])
        else:
            if np.random.rand() < 0.5:
                Q1_dql['A'][1] += alpha_b * (0 - Q1_dql['A'][1])
            else:
                Q2_dql['A'][1] += alpha_b * (0 - Q2_dql['A'][1])

    left_pct_ql += left_count_ql
    left_pct_dql += left_count_dql

left_pct_ql /= n_experiments
left_pct_dql /= n_experiments

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.convolve(left_pct_ql, np.ones(10)/10, mode='valid') * 100,
        'r-', linewidth=2, label='Q-Learning')
ax.plot(np.convolve(left_pct_dql, np.ones(10)/10, mode='valid') * 100,
        'b-', linewidth=2, label='Double Q-Learning')
ax.axhline(y=5, color='black', linestyle='--', alpha=0.5, label='Optimal (5% from ε)')
ax.set_xlabel('Episode')
ax.set_ylabel('% Left Actions')
ax.set_title('Maximization Bias: Q-Learning vs Double Q-Learning\n'
             '(Left is suboptimal — lower is better)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 6. Image Enhancement Environment

Now let's apply Q-learning to our image enhancement task.

In [ ]:
class ImageEnhancementEnv:
    """Tabular image enhancement MDP.

    State: (contrast, sharpness, noise) each in {0,...,n_levels-1}
    Actions: Image processing filters
    """

    def __init__(self, n_levels=5, max_steps=8):
        self.n_levels = n_levels
        self.max_steps = max_steps
        self.action_names = ['Sharpen', 'Denoise', 'Contrast+', 'Hist Eq', 'Edge Enh', 'Done']
        self.n_actions = len(self.action_names)
        self.state = None
        self.steps = 0

    def reset(self, start_state=None):
        if start_state is not None:
            self.state = list(start_state)
        else:
            self.state = [
                np.random.randint(0, 3),
                np.random.randint(0, 2),
                np.random.randint(2, self.n_levels)
            ]
        self.steps = 0
        return tuple(self.state)

    def step(self, action):
        self.steps += 1
        c, s, n = self.state

        if action == 5 or self.steps >= self.max_steps:
            quality = (c + s + (self.n_levels - 1 - n)) / (3 * (self.n_levels - 1))
            return tuple(self.state), quality * 5, True

        reward = -0.05

        if action == 0:  # Sharpen
            self.state[1] = min(s + 1, self.n_levels - 1)
            if np.random.rand() < 0.3:
                self.state[2] = min(n + 1, self.n_levels - 1)
            reward += 0.3 if s < self.n_levels - 1 else -0.1
        elif action == 1:  # Denoise
            self.state[2] = max(n - 1, 0)
            if np.random.rand() < 0.2:
                self.state[1] = max(s - 1, 0)
            reward += 0.3 if n > 0 else -0.1
        elif action == 2:  # Contrast+
            self.state[0] = min(c + 1, self.n_levels - 1)
            reward += 0.3 if c < self.n_levels - 1 else -0.2
        elif action == 3:  # Hist Eq
            self.state[0] = min(c + 1, self.n_levels - 1)
            if np.random.rand() < 0.5:
                self.state[2] = max(n - 1, 0)
            reward += 0.2
        elif action == 4:  # Edge Enhance
            self.state[1] = min(s + 2, self.n_levels - 1)
            self.state[2] = min(n + 1, self.n_levels - 1)
            reward += 0.1

        reward += np.random.normal(0, 0.05)
        return tuple(self.state), reward, False

img_env = ImageEnhancementEnv(n_levels=5, max_steps=8)
print(f"Image Enhancement Env: {img_env.n_levels}^3 = {img_env.n_levels**3} states")
print(f"Actions: {img_env.action_names}")

In [ ]:
# Train Q-learning agent
def train_q_learning(env, agent, n_episodes=30000):
    episode_rewards = []
    td_errors_ep = []
    epsilon_history = []

    for ep in range(n_episodes):
        state = env.reset()
        done = False
        ep_reward = 0
        ep_errors = []

        while not done:
            action = agent.get_action(state)
            next_state, reward, done = env.step(action)
            td_error = agent.update(state, action, reward, next_state, done)
            ep_errors.append(abs(td_error))
            ep_reward += reward
            state = next_state

        episode_rewards.append(ep_reward)
        td_errors_ep.append(np.mean(ep_errors) if ep_errors else 0)
        epsilon_history.append(agent.epsilon)
        agent.decay_epsilon()

    return episode_rewards, td_errors_ep, epsilon_history

# Train Q-learning and Double Q-learning
print("Training Q-Learning...")
ql_agent = QLearningAgent(img_env.n_actions, alpha=0.1, gamma=0.95)
ql_rewards, ql_errors, ql_epsilon = train_q_learning(img_env, ql_agent, n_episodes=30000)

print("Training Double Q-Learning...")
dql_agent = DoubleQLearningAgent(img_env.n_actions, alpha=0.1, gamma=0.95)
dql_rewards, dql_errors, dql_epsilon = train_q_learning(img_env, dql_agent, n_episodes=30000)

print("✅ Training complete!")

In [ ]:
# Plot training results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

window = 500

# Learning curves
ql_smooth = np.convolve(ql_rewards, np.ones(window)/window, mode='valid')
dql_smooth = np.convolve(dql_rewards, np.ones(window)/window, mode='valid')
axes[0, 0].plot(ql_smooth, 'r-', linewidth=2, label='Q-Learning')
axes[0, 0].plot(dql_smooth, 'b-', linewidth=2, label='Double Q-Learning')
axes[0, 0].set_xlabel('Episode')
axes[0, 0].set_ylabel('Episode Reward (smoothed)')
axes[0, 0].set_title('Learning Curves')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# TD Error
ql_err_smooth = np.convolve(ql_errors, np.ones(window)/window, mode='valid')
dql_err_smooth = np.convolve(dql_errors, np.ones(window)/window, mode='valid')
axes[0, 1].plot(ql_err_smooth, 'r-', linewidth=1.5, label='Q-Learning')
axes[0, 1].plot(dql_err_smooth, 'b-', linewidth=1.5, label='Double Q-Learning')
axes[0, 1].set_xlabel('Episode')
axes[0, 1].set_ylabel('Mean |TD Error|')
axes[0, 1].set_title('TD Error Convergence')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Epsilon decay
axes[1, 0].plot(ql_epsilon, 'r-', linewidth=2)
axes[1, 0].set_xlabel('Episode')
axes[1, 0].set_ylabel('ε')
axes[1, 0].set_title('Exploration Rate Decay')
axes[1, 0].grid(True, alpha=0.3)

# Q-value heatmap
sample_states = [(0,0,4), (1,1,3), (2,2,2), (3,3,1), (4,4,0),
                 (0,2,4), (2,0,3), (3,1,2)]
q_matrix = np.zeros((len(sample_states), img_env.n_actions))
for i, s in enumerate(sample_states):
    if s in ql_agent.Q:
        q_matrix[i] = ql_agent.Q[s]

im = axes[1, 1].imshow(q_matrix, cmap='RdYlGn', aspect='auto')
axes[1, 1].set_yticks(range(len(sample_states)))
axes[1, 1].set_yticklabels([str(s) for s in sample_states], fontsize=9)
axes[1, 1].set_xticks(range(img_env.n_actions))
axes[1, 1].set_xticklabels(img_env.action_names, rotation=30, ha='right', fontsize=9)
axes[1, 1].set_title('Q-Table Heatmap (Q-Learning)')
plt.colorbar(im, ax=axes[1, 1])

for i in range(len(sample_states)):
    for j in range(img_env.n_actions):
        axes[1, 1].text(j, i, f'{q_matrix[i,j]:.1f}', ha='center', va='center', fontsize=8)

plt.suptitle('Q-Learning for Image Enhancement', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visual demonstration: step-by-step image enhancement
def demonstrate_policy(env, agent, start_state, max_steps=10):
    """Run the learned policy and show each step."""
    state = env.reset(start_state)
    trajectory = [(state, None, 0)]
    total_reward = 0

    for step in range(max_steps):
        action = np.argmax(agent.Q[state])  # Greedy
        next_state, reward, done = env.step(action)
        total_reward += reward
        trajectory.append((next_state, action, reward))

        if done:
            break
        state = next_state

    return trajectory, total_reward

# Test from worst state
start = (0, 0, 4)
traj, total_r = demonstrate_policy(img_env, ql_agent, start)

print("Q-Learning: Optimal Image Enhancement Pipeline")
print("=" * 60)
print("Starting state: (Contrast=0, Sharpness=0, Noise=4) [worst]")
print()

for i, (state, action, reward) in enumerate(traj):
    if action is not None:
        c, s, n = state
        print(f"  Step {i}: Apply {img_env.action_names[action]:>12} → "
              f"(C={c}, S={s}, N={n}) | reward = {reward:+.3f}")
    else:
        c, s, n = state
        print(f"  Start:  (C={c}, S={s}, N={n})")

print(f"\n  Total reward: {total_r:.3f}")

# Visualize as bar chart progression
fig, axes = plt.subplots(1, len(traj), figsize=(3 * len(traj), 4))
if len(traj) == 1:
    axes = [axes]

categories = ['Contrast', 'Sharpness', '4-Noise']
colors_bar = ['#3498db', '#e74c3c', '#2ecc71']

for i, (state, action, reward) in enumerate(traj):
    c, s, n = state
    values = [c, s, 4 - n]  # Invert noise so higher = better
    axes[i].bar(categories, values, color=colors_bar, edgecolor='black')
    axes[i].set_ylim(0, 5)

    if action is not None:
        axes[i].set_title(f'← {img_env.action_names[action]}\nStep {i}', fontsize=10)
    else:
        axes[i].set_title('Initial\nStep 0', fontsize=10)

    if i > 0:
        axes[i].set_yticklabels([])
    else:
        axes[i].set_ylabel('Quality Level')

plt.suptitle('Step-by-Step Image Enhancement (Q-Learning Policy)', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Hyperparameter Analysis

In [ ]:
# Grid search over α and γ
alphas = [0.01, 0.05, 0.1, 0.2, 0.5]
gammas = [0.8, 0.9, 0.95, 0.99]

results_grid = np.zeros((len(gammas), len(alphas)))

for i, g in enumerate(gammas):
    for j, a in enumerate(alphas):
        agent = QLearningAgent(img_env.n_actions, alpha=a, gamma=g,
                               epsilon=1.0, epsilon_decay=0.999)
        rewards, _, _ = train_q_learning(img_env, agent, n_episodes=8000)
        results_grid[i, j] = np.mean(rewards[-2000:])

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(results_grid, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(alphas)))
ax.set_xticklabels([str(a) for a in alphas])
ax.set_yticks(range(len(gammas)))
ax.set_yticklabels([str(g) for g in gammas])
ax.set_xlabel('Learning Rate α')
ax.set_ylabel('Discount Factor γ')
ax.set_title('Q-Learning: Hyperparameter Grid Search\n(Average reward over last 2000 episodes)')

for i in range(len(gammas)):
    for j in range(len(alphas)):
        ax.text(j, i, f'{results_grid[i,j]:.2f}', ha='center', va='center',
                fontsize=11, fontweight='bold')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

best_idx = np.unravel_index(results_grid.argmax(), results_grid.shape)
print(f"\nBest hyperparameters: α={alphas[best_idx[1]]}, γ={gammas[best_idx[0]]}")
print(f"Best average reward: {results_grid[best_idx]:.3f}")

In [ ]:
# Compare all methods on image enhancement
print("\nComparing Q-Learning, Double Q-Learning, and SARSA...")

n_eps = 25000
methods = {
    'Q-Learning': lambda: QLearningAgent(img_env.n_actions, alpha=0.1, gamma=0.95),
    'Double Q-Learning': lambda: DoubleQLearningAgent(img_env.n_actions, alpha=0.1, gamma=0.95),
    'SARSA': lambda: SARSAAgent(img_env.n_actions, alpha=0.1, gamma=0.95),
}

all_rewards = {}
for name, agent_fn in methods.items():
    agent = agent_fn()
    if name == 'SARSA':
        ep_rewards = []
        for ep in range(n_eps):
            state = img_env.reset()
            action = agent.get_action(state)
            done = False
            ep_r = 0
            while not done:
                next_state, reward, done = img_env.step(action)
                next_action = agent.get_action(next_state)
                agent.update(state, action, reward, next_state, next_action, done)
                state, action = next_state, next_action
                ep_r += reward
            ep_rewards.append(ep_r)
            agent.decay_epsilon()
        all_rewards[name] = ep_rewards
    else:
        rewards, _, _ = train_q_learning(img_env, agent, n_episodes=n_eps)
        all_rewards[name] = rewards

fig, ax = plt.subplots(figsize=(14, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']
window = 500

for (name, rewards), color in zip(all_rewards.items(), colors):
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
    ax.plot(smoothed, color=color, linewidth=2, label=name)

ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward (smoothed)')
ax.set_title('Method Comparison on Image Enhancement Task')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFinal performance (avg reward, last 2000 episodes):")
for name, rewards in all_rewards.items():
    print(f"  {name:25s}: {np.mean(rewards[-2000:]):.3f}")

---

## 📝 Summary

In this notebook, we covered:

1. **Q-Learning Derivation** — from Bellman optimality to the off-policy TD update using $\max_a Q(S', a)$
2. **Convergence Proof Sketch** — via contraction mapping and stochastic approximation
3. **SARSA vs Q-Learning** — on-policy learns safe paths, off-policy learns optimal paths (Cliff Walking demo)
4. **Maximization Bias** — $\mathbb{E}[\max Q] \geq \max \mathbb{E}[Q]$; Double Q-Learning corrects this
5. **Image Enhancement** — tabular Q-learning finds optimal filter sequences from degraded images

### 🔑 Key Equations

| Concept | Formula |
|---------|----------|
| Q-Learning | $Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha[R_{t+1} + \gamma \max_a Q(S_{t+1},a) - Q(S_t,A_t)]$ |
| SARSA | $Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)]$ |
| Bellman Optimality | $q_*(s,a) = \mathbb{E}[R + \gamma \max_{a'} q_*(S', a') | s, a]$ |
| Max Bias | $\mathbb{E}[\max_a Q(s,a)] \geq \max_a \mathbb{E}[Q(s,a)]$ |

### On-Policy vs Off-Policy Summary

| Property | SARSA (On-Policy) | Q-Learning (Off-Policy) |
|----------|------------------|-------------------------|
| Learns | $q_\pi$ (current policy) | $q_*$ (optimal) |
| Update target | $Q(S', A')$ | $\max_a Q(S', a)$ |
| Behavior = target? | Yes | No |
| Safety-aware? | Yes (accounts for exploration) | No (assumes greedy execution) |

---

## ➡️ Next

In the next module, we'll move to **Deep RL methods** that can handle continuous state spaces — where the image itself (not a discretized quality vector) becomes the state. We'll cover Deep Q-Networks (DQN), Policy Gradient methods, and Actor-Critic architectures.